# 03 — Data Cleaning & EDA · Banana Ripeness Detection (YOLO26s)

**หน้าที่ของ notebook นี้ (MASTER PROMPT ข้อ 5 — ต้องมี report ทุกขั้น)**
1. **EDA**: ขนาดภาพ / จำนวน bbox ต่อภาพ / ขนาด-สัดส่วน-ตำแหน่ง bbox แยกตาม source + grid ตัวอย่างภาพต่อคลาส
2. **ภาพเบลอ**: Variance of Laplacian **ในกล่อง bbox** เทียบก่อน/หลังเบลอซ้ำ (blur ratio — ไม่ลำเอียงตาม texture ของผิวกล้วย) → threshold จาก percentile ของแต่ละ source + ตัวอย่างภาพที่ถูกตัด
3. **ภาพซ้ำ**: pHash + Hamming distance **+ สีของวัตถุ + label** → ซ้ำ = โครงสร้างเหมือน และ (สีเหมือน หรือ label เหมือน) → เหลือภาพที่คมที่สุด 1 ภาพ; คู่ที่ทั้งสีและ label ต่าง (คนละระยะสุก) เก็บไว้ + flag; บันทึก "scene group" ให้ nb04 กัน leakage
4. **ตรวจ label**: bbox เล็กผิดปกติ, bbox ซ้อนกันคลาสเดียวกัน (annotation ซ้ำ), ซ้อนกันคนละคลาส (ขัดแย้ง)
5. **ภาพที่ตัดสินความสุกไม่ได้**: วัดบน pixel ของกล้วย (ในกล่อง bbox หักพื้นหลัง) — มืดเกิน / ขาวโพลน / ซีด / contrast ต่ำ, และกล้วยเล็กเกินไปในภาพ
6. **Class balance** ก่อน/หลัง + เสนอ oversampling ratio → เขียน `data/processed/` (Mendeley ย่อเหลือด้านยาว 1,280 px)
   + `reports/data_cleaning_report.csv` + `reports/before_after_summary.md` + `reports/manifest_clean.csv`

หลักการตั้ง threshold: **percentile ของข้อมูลจริง (adaptive ต่อ source) แล้ว cap ด้วยค่าที่มีเหตุผลทางกายภาพ** — เพื่อไม่ให้ percentile ตัดภาพดีทิ้งเมื่อทั้งชุดคุณภาพดีอยู่แล้ว

> kernel: **ML Environment (ds69) + YOLO26** · ต้องรัน nb01, nb02 ก่อน


In [ ]:
# === CELL 1 === โหลด config + manifest จาก nb02 + รวมค่าตั้งของ cleaning ไว้ที่เดียว
# ทำอะไร : อ่าน configs/project.yaml, reports/manifest_interim.csv, src/viz.py และกำหนด threshold แบบ percentile ทั้งหมดใน NB03
# ทำไม   : ทุกเกณฑ์การตัดต้องอยู่ที่เดียว ตรวจสอบ/ปรับได้ และรายงานได้ว่าใช้ค่าอะไร (spec ข้อ 5: ห้าม hardcode ลอย ๆ)
import os, sys, re, json, shutil, time, random, importlib, math
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
import yaml
import numpy as np
import pandas as pd
import cv2
from PIL import Image, ImageDraw
import imagehash
from tqdm import tqdm
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
CFG_PATH = PROJECT_ROOT / "configs" / "project.yaml"
assert CFG_PATH.exists(), f"ไม่พบ {CFG_PATH} → รัน 01_setup_env.ipynb ก่อน"
with open(CFG_PATH, encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)
DIRS = {k: PROJECT_ROOT / v for k, v in CONFIG["dirs"].items()}
SEED = CONFIG["seed"]
random.seed(SEED)
np.random.seed(SEED)
CLASSES = {int(k): v for k, v in CONFIG["classes"].items()}
NAMES = list(CLASSES.values())
NC = len(CLASSES)

sys.path.insert(0, str(DIRS["src"]))
import viz
importlib.reload(viz)
viz.apply_style()
CLASS_RGB = viz.class_colors(CONFIG)                      # สีประจำคลาส (0–1) สำหรับวาด bbox ในกราฟ
CLASS_RGB255 = {k: tuple(int(c * 255) for c in v) for k, v in CLASS_RGB.items()}

IM_IN, LB_IN = DIRS["data_interim"] / "images", DIRS["data_interim"] / "labels"
IM_OUT, LB_OUT = DIRS["data_processed"] / "images", DIRS["data_processed"] / "labels"
MANIFEST_IN = DIRS["reports"] / "manifest_interim.csv"
assert MANIFEST_IN.exists(), "ไม่พบ reports/manifest_interim.csv → รัน 02_data_merge_mapping.ipynb ก่อน"
df = pd.read_csv(MANIFEST_IN)
n_files = sum(1 for p in IM_IN.iterdir() if p.is_file())
assert len(df) == n_files, f"manifest {len(df)} แถว แต่ interim มี {n_files} ไฟล์ → รัน nb02 ใหม่ให้ตรงกัน"

# --- ค่าตั้งทั้งหมดของ nb03 --------------------------------------------------------------------
# กติกา: threshold = percentile ของ source นั้น ๆ แล้ว "cap" ด้วยค่าเชิงกายภาพ → ถ้าทั้งชุดคุณภาพดี percentile จะไม่ตัดภาพดีทิ้ง
NB03 = {
    "analysis_long_side": 640,   # ย่อภาพให้ด้านยาว 640 ก่อนวัด blur/brightness/hash → Mendeley 12MP กับ Roboflow 640px เทียบกันได้
    "blur":     {"percentile": 3.0, "ratio_cap": 2.0, "sigma": 1.5},   # blur_ratio = VoL(ในกล่อง) / VoL(ในกล่อง หลังเบลอ σ=1.5 ที่ 640px)
                                                                        # ภาพคม 4–20, เบลอจริง < 2 (วัดจากข้อมูลจริง + เบลอสังเคราะห์) → ตัด < min(p3, 2.0)
    # pHash วัด "โครงสร้าง" แบบ grayscale (มองไม่เห็นสี) → กติกาซ้ำ: hash ≤2 และ (สีวัตถุเหมือน ≤20 หรือ label เหมือน)
    # - สีเหมือน            → ภาพเดิม/burst           → ซ้ำ ลบ
    # - สีต่าง label เหมือน → ภาพเดิมที่ถูกเลื่อนสี (WB/hue augment ที่ติดมากับชุดข้อมูล; ตรวจพบจริงในรอบ v4) → ซ้ำ ลบ (สีปลอมกับ label เดิม = อันตรายกับงานที่ใช้สีตัดสิน)
    # - สีต่าง label ต่าง   → กล้วยเดิมคนละระยะสุก (หรือ label noise) → เก็บทั้งคู่ + flag ให้ตรวจ, อยู่ scene_group เดียวกัน
    "dup":      {"hash_size": 8, "exact_max_dist": 2, "color_max_dist": 20.0, "scene_max_dist": 8},   # ≤8 = ซีนเดียวกัน → split เดียวกันใน nb04
    "label":    {"tiny_area_percentile": 1.0, "tiny_area_cap": 0.001, "tiny_min_side_px": 6,   # bbox เล็กผิดปกติ
                 "dup_iou": 0.90, "conflict_iou": 0.70},                                        # ซ้อนคลาสเดียวกัน / คนละคลาส
    # exposure วัดบน "pixel ของวัตถุ" = pixel ในกล่อง bbox ที่สีต่างจากพื้นหลัง (median ของขอบภาพ) — ไม่ใช่ทั้งภาพ/ทั้งกล่อง
    # เพราะกล่องสี่เหลี่ยมรอบกล้วยที่วางเฉียงมีพื้นหลังขาว > 50% → ถ้าวัดทั้งกล่อง กล้วยดิบเขียวคม ๆ จะถูกตัดว่า "ขาวโพลน" (ตรวจพบและแก้แล้ว)
    "exposure": {"bg_tolerance": 45,                                # |RGB − สีพื้นหลัง| (L1) ≤ 45 ถือเป็นพื้นหลัง ไม่นับ (+ pixel ≥235 ทุกช่อง ถ้าพื้นหลังสว่าง)
                 "dark_percentile": 1.0, "dark_cap": 40.0,          # brightness ของวัตถุ (0–255) < min(p1, 40) → กล้วยมืดเกินจะดูสี
                 "sat_percentile": 99.0, "sat_floor": 0.30,         # สัดส่วน pixel วัตถุที่ ≥ 250 > max(p99, 0.30) → ขาวโพลน (over-exposed)
                 "bright_percentile": 99.5, "bright_floor": 215.0,  # brightness ของวัตถุ > max(p99.5, 215) → ซีดจนสีเพี้ยน
                 "contrast_percentile": 1.0, "contrast_cap": 12.0,  # std ของ gray ของวัตถุ < min(p1, 12) → แบน/ไม่มีรายละเอียด
                 "smallest_box_percentile": 1.0, "smallest_box_cap": 0.002},  # bbox ใหญ่สุดในภาพ < min(p1, 0.2% ของภาพ) → กล้วยเล็กเกินจะดูสี
    "processed": {"mendeley_max_side": 1280, "jpeg_quality": 92},   # ย่อ Mendeley (4000px) ให้เทรนเร็ว; label เป็นค่า normalized ไม่ต้องแก้
    "oversample_cap": 3.0,        # เพดาน oversampling ratio ที่จะเสนอให้ nb05
}
METRICS_CACHE = DIRS["reports"] / "03_image_metrics.csv"
RUN = {"notebook": "03_data_cleaning_eda", "started_at": datetime.now().isoformat(timespec="seconds"), "thresholds_used": {}}

def parse_label(path: Path):
    rows = []
    if not Path(path).exists():
        return rows
    for ln in Path(path).read_text(encoding="utf-8", errors="ignore").splitlines():
        p = ln.split()
        if len(p) >= 5:
            rows.append((int(float(p[0])), float(p[1]), float(p[2]), float(p[3]), float(p[4])))
    return rows

BOXES = {r.file: parse_label(LB_IN / r.label) for r in df.itertuples()}   # file → [(cls, x, y, w, h), ...]
src_counts = {k: int(v) for k, v in df.source.value_counts().items()}
print(f"[OK] manifest_interim: {len(df):,} ภาพ | boxes {sum(len(v) for v in BOXES.values()):,} | sources = {src_counts}")
print(f"[OK] thresholds (percentile → cap): blur_ratio p{NB03['blur']['percentile']:g}/cap {NB03['blur']['ratio_cap']} | dup ≤{NB03['dup']['exact_max_dist']} (scene ≤{NB03['dup']['scene_max_dist']}) | "
      f"exposure in-box p{NB03['exposure']['dark_percentile']:g}/p{NB03['exposure']['sat_percentile']:g}")


In [ ]:
# === CELL 2 === EDA: ขนาดภาพ / bbox ต่อภาพ / ขนาด-สัดส่วน-ตำแหน่ง bbox + grid ตัวอย่างภาพต่อคลาสต่อ source
# ทำอะไร : สร้างตาราง bbox-level แล้ววาดกราฟภาพรวม 4 ช่อง + ตารางตัวอย่างภาพ (วาด bbox สีประจำคลาส)
# ทำไม   : ต้องรู้ว่า 2 แหล่งข้อมูลต่างกันแค่ไหน (Mendeley = กล้วยเดี่ยว ภาพใหญ่ / Roboflow = หลายผล ภาพเล็ก) ก่อนตั้งเกณฑ์ cleaning
box_rows = []
for r in df.itertuples():
    for c, x, y, w, h in BOXES[r.file]:
        box_rows.append({"file": r.file, "source": r.source, "cls": c, "x": x, "y": y, "w": w, "h": h,
                         "area_ratio": w * h, "w_px": w * r.width, "h_px": h * r.height,
                         "aspect": (w * r.width) / (h * r.height) if h > 0 else np.nan})
bx = pd.DataFrame(box_rows)
df["n_boxes"] = df["file"].map(lambda f: len(BOXES[f]))

print("— bbox ต่อภาพ —")
print(pd.crosstab(df["source"], df["n_boxes"].clip(upper=6).astype(str).replace("6", "6+"), margins=True))
print("\n— area ratio ของ bbox (สัดส่วนพื้นที่ภาพ) —")
print(bx.groupby("source")["area_ratio"].describe(percentiles=[.01, .05, .5, .95]).round(4))
print("\n— ขนาดภาพ (px) —")
print(df.groupby("source")[["width", "height"]].agg(["min", "median", "max"]).round(0))

def load_tile(file, long_side=256, draw_boxes=True, boxes=None):
    # โหลดภาพจาก interim ย่อให้ด้านยาว = long_side แล้ววาด bbox (สีตามคลาส) — ใช้ทำ grid ทุกที่ใน notebook นี้
    with Image.open(IM_IN / file) as im:
        im = im.convert("RGB")
        s = long_side / max(im.size)
        im = im.resize((max(1, round(im.width * s)), max(1, round(im.height * s))))
    if draw_boxes:
        d = ImageDraw.Draw(im)
        for c, x, y, w, h in (boxes if boxes is not None else BOXES[file]):
            W, H = im.size
            d.rectangle([(x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H], outline=CLASS_RGB255[c], width=3)
    return im

def image_grid(files, titles, ncols, path, tile=256, boxes_by_file=None, suptitle=None):
    # วาง tile เป็นตาราง บันทึกเป็น PNG dpi 300 (spec ข้อ 11) — คืน path
    n = len(files)
    if n == 0:
        return None
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(2.6 * ncols, 2.6 * nrows + (0.4 if suptitle else 0)))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, f, t in zip(axes, files, titles):
        ax.imshow(load_tile(f, tile, boxes=(boxes_by_file or {}).get(f)))
        ax.set_title(t, fontsize=7, loc="left")
    if suptitle:
        fig.suptitle(suptitle, x=0.01, ha="left", fontsize=11)
    fig.tight_layout()
    out = viz.save_fig(fig, DIRS["figures"], path)
    plt.show()
    plt.close(fig)
    return out

# --- กราฟภาพรวม 4 ช่อง ---
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for src in ("mendeley", "roboflow"):
    g = df[df.source == src]
    axes[0, 0].scatter(g["width"], g["height"], s=6, alpha=0.5, color=viz.SOURCE_COLORS[src], label=src.capitalize())
axes[0, 0].set(title="Image size (px)", xlabel="width", ylabel="height")
axes[0, 0].set_title("Image size (px)", loc="left")
axes[0, 0].legend()
nb_tab = pd.crosstab(df["n_boxes"].clip(upper=6), df["source"])
nb_tab.plot.bar(ax=axes[0, 1], color=[viz.SOURCE_COLORS[c] for c in nb_tab.columns], width=0.7)
axes[0, 1].set_title("Boxes per image", loc="left")
axes[0, 1].set(xlabel="boxes per image (6 = 6+)", ylabel="images")
axes[0, 1].tick_params(axis="x", rotation=0)
for src in ("mendeley", "roboflow"):
    axes[1, 0].hist(np.log10(bx.loc[bx.source == src, "area_ratio"].clip(lower=1e-5)), bins=40, alpha=0.6,
                    color=viz.SOURCE_COLORS[src], label=src.capitalize())
axes[1, 0].set_title("Box area ratio (log10 of box area / image area)", loc="left")
axes[1, 0].set(xlabel="log10(area ratio)", ylabel="boxes")
axes[1, 0].legend()
hb = axes[1, 1].hist2d(bx["x"], bx["y"], bins=20, range=[[0, 1], [0, 1]], cmap="Blues")
axes[1, 1].invert_yaxis()
axes[1, 1].set_title("Box centre positions (all boxes)", loc="left")
axes[1, 1].set(xlabel="x centre", ylabel="y centre")
fig.colorbar(hb[3], ax=axes[1, 1], label="boxes")
fig.suptitle(f"EDA — {len(df):,} images, {len(bx):,} boxes (interim, before cleaning)", x=0.01, ha="left", fontsize=12)
fig.tight_layout()
out = viz.save_fig(fig, DIRS["figures"], "03_eda_overview.png")
plt.show()
print(f"[OK] {out.relative_to(PROJECT_ROOT)}")

# --- grid ตัวอย่าง: แถว = คลาส, คอลัมน์ = 3 ภาพ Mendeley + 3 ภาพ Roboflow ---
rng = random.Random(SEED)
files, titles = [], []
for c, name in CLASSES.items():
    for src in ("mendeley", "roboflow"):
        pool = df[(df.source == src) & (df.dominant_class == c)]["file"].tolist()
        picks = rng.sample(pool, min(3, len(pool)))
        for i in range(3):
            if i < len(picks):
                files.append(picks[i])
                titles.append(f"{name} · {src[:3]} · {picks[i][:22]}")
            else:
                files.append(None)
                titles.append(f"{name} · {src[:3]} · (no image)")
# ช่องว่างสำหรับคลาสที่ source นั้นไม่มี (Mendeley ไม่มี semi_ripe)
nrows, ncols = NC, 6
fig, axes = plt.subplots(nrows, ncols, figsize=(2.4 * ncols, 2.4 * nrows + 0.4))
for ax, f, t in zip(axes.reshape(-1), files, titles):
    ax.axis("off")
    if f is not None:
        ax.imshow(load_tile(f, 224))
    ax.set_title(t, fontsize=6.5, loc="left")
fig.suptitle("Samples per class — left 3: Mendeley, right 3: Roboflow (box colour = class)", x=0.01, ha="left", fontsize=11)
fig.tight_layout()
out = viz.save_fig(fig, DIRS["figures"], "03_samples_by_class.png")
plt.show()
plt.close(fig)
print(f"[OK] {out.relative_to(PROJECT_ROOT)}")

# --- ภาพ Mendeley ที่ชื่อไฟล์กับ label ไม่ตรงกัน (จาก nb02) — ให้ดูด้วยตา ---
summary02 = DIRS["reports"] / "02_merge_summary.json"
mm_files = []
if summary02.exists():
    with open(summary02, encoding="utf-8") as f:
        notes = json.load(f).get("mendeley_notes", {}).get("filename_label_mismatch", [])
    for line in notes:
        orig = line.split(":")[0].strip()
        hit = df[(df.source == "mendeley") & (df.orig_name == orig)]["file"].tolist()
        mm_files += hit
if mm_files:
    out = image_grid(mm_files, [f"{f} · label={CLASSES[BOXES[f][0][0]]}" for f in mm_files], ncols=min(3, len(mm_files)),
                     path="03_mendeley_name_vs_label.png", suptitle="Mendeley: filename says overripe, label says ripe — kept label as-is")
    print(f"[OK] {out.relative_to(PROJECT_ROOT)} ({len(mm_files)} ภาพ)")
RUN["eda"] = {"images": int(len(df)), "boxes": int(len(bx)), "boxes_per_image_mean": round(float(df.n_boxes.mean()), 3),
              "area_ratio_median": {s: round(float(bx.loc[bx.source == s, "area_ratio"].median()), 4) for s in ("mendeley", "roboflow")}}


In [ ]:
# === CELL 3 === วัดคุณภาพภาพทุกใบ (ในกล่อง bbox): blur ratio / brightness / contrast / saturation + pHash แล้วตัดภาพเบลอ
# ทำอะไร : ย่อภาพให้ด้านยาว 640 → วัดเฉพาะบริเวณ bbox (ตัวกล้วย): VoL, VoL หลังเบลอ σ=1.5 → blur_ratio, mean/std ของ gray,
#          สัดส่วน pixel ที่ ≥250 (ขาวโพลน); pHash ทั้งภาพ; cache ลง reports/03_image_metrics.csv → threshold blur = min(p3 ของ source, 2.0)
# ทำไม   : (1) VoL ทั้งภาพบนภาพสตูดิโอวัด "texture" ไม่ใช่ "ความคม" — กล้วยดิบผิวเรียบได้ค่าต่ำสุดเสมอ → ถ้าใช้จะตัดคลาส unripe ทิ้งอย่างลำเอียง
#          (2) blur_ratio (เบลอซ้ำแล้ว VoL ตกแค่ไหน) แทบไม่ขึ้นกับ texture: ภาพคมได้ 4–20, ภาพเบลอจริง < 2 (ตรวจกับข้อมูลจริง + เบลอสังเคราะห์แล้ว)
#          (3) พื้นหลังขาวของ Mendeley ทำให้ brightness ทั้งภาพสว่างเสมอ → ต้องวัดในกล่อง
def analyze_image(path: Path, boxes) -> dict:
    with Image.open(path) as im:
        im = im.convert("RGB")
        s = NB03["analysis_long_side"] / max(im.size)
        if s < 1:
            im = im.resize((max(1, round(im.width * s)), max(1, round(im.height * s))), Image.BILINEAR)
        gray = cv2.cvtColor(np.asarray(im), cv2.COLOR_RGB2GRAY)
        H, W = gray.shape
        mask = np.zeros((H, W), dtype=bool)
        for c, x, y, w, h in boxes:
            mask[max(0, int((y - h / 2) * H)):min(H, int(math.ceil((y + h / 2) * H))),
                 max(0, int((x - w / 2) * W)):min(W, int(math.ceil((x + w / 2) * W)))] = True
        if mask.sum() < 16:                       # กล่องเล็กจนวัดไม่ได้ → ใช้ทั้งภาพ
            mask[:] = True
        lap = cv2.Laplacian(gray, cv2.CV_64F)
        lap_blur = cv2.Laplacian(cv2.GaussianBlur(gray, (0, 0), NB03["blur"]["sigma"]), cv2.CV_64F)
        vol_box = float(lap[mask].var())
        # pixel ของวัตถุ = ในกล่อง และสีต่างจากพื้นหลัง (median ของกรอบขอบภาพ 5%) เกิน bg_tolerance
        arr = np.asarray(im)
        bw = max(2, int(0.05 * min(H, W)))
        border = np.concatenate([arr[:bw].reshape(-1, 3), arr[-bw:].reshape(-1, 3), arr[:, :bw].reshape(-1, 3), arr[:, -bw:].reshape(-1, 3)])
        bg = np.median(border, axis=0)
        bg_mask = np.abs(arr.astype(np.int16) - bg.astype(np.int16)).sum(axis=2) <= NB03["exposure"]["bg_tolerance"]
        if bg.mean() > 200:                       # พื้นหลังสว่าง (กระดาษขาว/โต๊ะขาว): pixel ขาวเกือบสุดในกล่องคือพื้นหลังที่โดนแสงไม่เท่ากัน ไม่ใช่กล้วย
            bg_mask |= arr.min(axis=2) >= 235
        obj = mask & ~bg_mask
        if obj.sum() < 64:                        # แยกวัตถุจากพื้นหลังไม่ได้ (พื้นหลังไม่เรียบ) → ใช้ทั้งกล่อง
            obj = mask
        g = gray[obj]
        rgb = arr[obj].mean(axis=0)               # สีเฉลี่ยของวัตถุ (ใช้แยก "ภาพเดิม" ออกจาก "กล้วยพวงเดิมคนละวัน" ตอน dedup)
        return {"vol": float(lap.var()), "vol_box": vol_box,
                "obj_r": float(rgb[0]), "obj_g": float(rgb[1]), "obj_b": float(rgb[2]),
                "blur_ratio": float(vol_box / max(lap_blur[mask].var(), 1e-6)),
                "brightness": float(gray.mean()), "contrast": float(gray.std()),
                "obj_frac": float(obj.sum() / mask.sum()),
                "brightness_obj": float(g.mean()), "contrast_obj": float(g.std()),
                "sat_frac_obj": float((g >= 250).mean()), "dark_frac_obj": float((g <= 10).mean()),
                "phash": str(imagehash.phash(im, hash_size=NB03["dup"]["hash_size"]))}

METRIC_COLS = ["vol", "vol_box", "blur_ratio", "brightness", "contrast", "obj_frac", "brightness_obj", "contrast_obj", "sat_frac_obj", "dark_frac_obj",
               "obj_r", "obj_g", "obj_b", "phash"]
metrics = None
if METRICS_CACHE.exists():
    cached = pd.read_csv(METRICS_CACHE, dtype={"phash": str})
    if set(cached["file"]) == set(df["file"]) and all(c in cached.columns for c in METRIC_COLS):
        metrics = cached
        print(f"[SKIP] ใช้ metrics จาก cache {METRICS_CACHE.name} ({len(metrics):,} ภาพ)")
if metrics is None:
    rows = []
    for f in tqdm(df["file"], desc="analyze images"):
        rows.append({"file": f, **analyze_image(IM_IN / f, BOXES[f])})
    metrics = pd.DataFrame(rows)
    metrics.to_csv(METRICS_CACHE, index=False)
    print(f"[OK] คำนวณ metrics {len(metrics):,} ภาพ → {METRICS_CACHE.relative_to(PROJECT_ROOT)}")
df = df.drop(columns=[c for c in METRIC_COLS if c in df.columns]).merge(metrics, on="file", how="left")
assert df[METRIC_COLS].notna().all().all(), "metrics บางภาพหาย"

print("\n— blur_ratio ต่อ source (สูง = คม) —")
print(df.groupby("source")["blur_ratio"].describe(percentiles=[.01, .03, .05, .10, .50]).round(2))

blur_thr = {}
for src in ("mendeley", "roboflow"):
    p = float(np.percentile(df.loc[df.source == src, "blur_ratio"], NB03["blur"]["percentile"]))
    blur_thr[src] = min(p, NB03["blur"]["ratio_cap"])
    print(f"[THRESHOLD] blur {src:<9}: p{NB03['blur']['percentile']:g} = {p:6.2f} | cap = {NB03['blur']['ratio_cap']} → ใช้ {blur_thr[src]:.2f}"
          + ("  (cap ชนะ → ชุดนี้คมพอ ตัดเฉพาะที่เบลอจริง)" if p > NB03["blur"]["ratio_cap"] else "  (percentile ชนะ)"))
df["blur_flag"] = df["blur_ratio"] < df["source"].map(blur_thr)
RUN["thresholds_used"]["blur_ratio"] = blur_thr

# --- กราฟ distribution + เส้น threshold ---
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for ax, src in zip(axes, ("mendeley", "roboflow")):
    v = df.loc[df.source == src, "blur_ratio"].clip(upper=30)
    ax.hist(v, bins=60, color=viz.SOURCE_COLORS[src], alpha=0.85)
    ax.axvline(blur_thr[src], color=viz.TEXT["primary"], ls="--", lw=1.2)
    n_cut = int(df[(df.source == src) & df.blur_flag].shape[0])
    ax.set_title(f"{src.capitalize()} — blur ratio in box (VoL / VoL after σ=1.5 blur), threshold {blur_thr[src]:.2f} → remove {n_cut}", loc="left", fontsize=9.5)
    ax.set(xlabel="blur ratio (clipped at 30; sharp = high)", ylabel="images")
fig.tight_layout()
out = viz.save_fig(fig, DIRS["figures"], "03_blur_distribution.png")
plt.show()
plt.close(fig)

# --- ตัวอย่างภาพที่ถูกตัด (เบลอที่สุดก่อน) ---
ex = df[df.blur_flag].sort_values("blur_ratio").groupby("source").head(6)
out2 = image_grid(ex["file"].tolist(), [f"{r.source[:3]} ratio={r.blur_ratio:.2f}" for r in ex.itertuples()], ncols=6,
                  path="03_blur_removed_examples.png", suptitle="Blur check — images flagged for removal (lowest blur ratio first)")
print(f"[OK] blur flagged: {int(df.blur_flag.sum())} ภาพ ({ {k: int(v) for k, v in df[df.blur_flag].source.value_counts().items()} }) → {out.name}" + (f", {out2.name}" if out2 else ""))
RUN["blur"] = {"flagged": int(df.blur_flag.sum()), "by_source": {k: int(v) for k, v in df[df.blur_flag].source.value_counts().items()}}


In [ ]:
# === CELL 4 === ภาพซ้ำ: pHash + Hamming distance + สีวัตถุ + label → ลบ near-duplicate + จัดกลุ่ม "ซีนเดียวกัน" ให้ nb04
# ทำอะไร : คำนวณระยะ Hamming ระหว่าง hash ทุกคู่ (5k × 5k) → คู่ที่ hash ≤ exact_max_dist ถือว่าซ้ำเมื่อ "สีวัตถุเหมือน" หรือ "label เหมือน"
#          (เก็บภาพที่คมสุดในกลุ่ม 1 ภาพ) | คู่ที่โครงสร้างเหมือน แต่ทั้งสีและ label ต่าง = กล้วยเดิมคนละระยะสุก → เก็บทั้งคู่ + flag review
#          ระยะ ≤ scene_max_dist (ไม่สนสี) = ซีน/burst/time-lapse เดียวกัน → scene_group เดียวกัน เพื่อให้ nb04 วางไว้ split เดียวกัน
# ทำไม   : pHash มองไม่เห็นสี — ชุดข้อมูลจริงมี "ภาพเดิมที่ถูกเลื่อนสี" (white balance/hue) จำนวนมากโดย label ไม่เปลี่ยน (ตรวจพบ 2026-09-19:
#          คู่ hash เท่ากันแต่สีต่าง 1,113 คู่ มีเพียง ~3% ที่คลาสต่าง) → ภาพพวกนี้คือสำเนา ไม่ใช่ข้อมูลใหม่ และสีที่ถูกเลื่อนโดย label เดิม
#          เป็นพิษกับงานที่ใช้สีตัดสินความสุก จึงลบ; ส่วน leakage: ภาพซีนเดียวกันต้องอยู่ split เดียวกัน — spec ข้อ 4 บังคับกัน
hashes = np.array([int(h, 16) for h in df["phash"]], dtype=np.uint64)
n = len(hashes)
POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def hamming_to_all(i):
    x = np.bitwise_xor(hashes[i], hashes)
    return POPCOUNT[x.view(np.uint8)].reshape(n, 8).sum(axis=1)

parent_exact, parent_scene = list(range(n)), list(range(n))
def find(par, i):
    while par[i] != i:
        par[i] = par[par[i]]
        i = par[i]
    return i
def union(par, a, b):
    ra, rb = find(par, a), find(par, b)
    if ra != rb:
        par[max(ra, rb)] = min(ra, rb)

D_EXACT, D_SCENE, C_MAX = NB03["dup"]["exact_max_dist"], NB03["dup"]["scene_max_dist"], NB03["dup"]["color_max_dist"]
rgb = df[["obj_r", "obj_g", "obj_b"]].to_numpy(dtype=float)
sig = df["file"].map(lambda f: tuple(sorted(Counter(b[0] for b in BOXES[f]).items())))   # label signature ของภาพ เช่น ((2,1),(3,2))
sig_arr = sig.to_numpy()
nn_dist, nn_idx, cross_pairs = np.full(n, 99, dtype=int), np.full(n, -1, dtype=int), 0
pair_stats = Counter()          # same_color / color_shift_same_label / kept_stage_change
kept_stage = []                 # คู่ที่เก็บไว้ทั้งคู่ (สีต่าง + label ต่าง)
src_arr = df["source"].to_numpy()
for i in tqdm(range(n), desc="hamming"):
    d = hamming_to_all(i)
    d[i] = 99
    j = int(d.argmin())
    nn_dist[i], nn_idx[i] = d[j], j
    close = np.where(d <= D_SCENE)[0]
    for j2 in close[close > i]:
        union(parent_scene, i, int(j2))
        if d[j2] <= D_EXACT:
            cdist = float(np.linalg.norm(rgb[i] - rgb[j2]))
            same_label = sig_arr[i] == sig_arr[j2]
            if cdist <= C_MAX or same_label:
                union(parent_exact, i, int(j2))
                pair_stats["same_color" if cdist <= C_MAX else "color_shift_same_label"] += 1
                if src_arr[i] != src_arr[j2]:
                    cross_pairs += 1
            else:
                pair_stats["kept_stage_change"] += 1
                kept_stage.append((i, int(j2), int(d[j2]), cdist))
df["nn_dist"], df["nn_file"] = nn_dist, df["file"].to_numpy()[nn_idx]
stage_idx = {k for pr in kept_stage for k in pr[:2]}
df["stage_pair_flag"] = [i in stage_idx for i in range(n)]    # ภาพที่มีคู่โครงสร้างเหมือน/สีต่าง/label ต่าง → ให้ตรวจด้วยตา (ไม่ลบ)
df["exact_cluster"] = [find(parent_exact, i) for i in range(n)]
df["scene_group"] = [find(parent_scene, i) for i in range(n)]

# ในแต่ละกลุ่มซ้ำ เก็บภาพที่ VoL สูงสุด (คมสุด) ไว้ 1 ภาพ ที่เหลือ flag
keep_idx = df.sort_values("blur_ratio", ascending=False).groupby("exact_cluster").head(1).index   # เก็บภาพที่คมสุดในกลุ่ม
df["dup_flag"] = ~df.index.isin(keep_idx)
n_clusters = int((df.groupby("exact_cluster").size() > 1).sum())
n_scene = int((df.groupby("scene_group").size() > 1).sum())
n_pairs_hash = sum(pair_stats.values())
print(f"[OK] nearest-neighbour distance: median {np.median(nn_dist):.0f} | ≤{D_EXACT}: {(nn_dist <= D_EXACT).sum():,} ภาพ | ≤{D_SCENE}: {(nn_dist <= D_SCENE).sum():,} ภาพ")
print(f"[OK] คู่ที่ hash ≤{D_EXACT}: {n_pairs_hash:,} คู่ = สีเหมือน (ภาพเดิม/burst) {pair_stats['same_color']:,} + สีถูกเลื่อนแต่ label เดิม (สำเนา) "
      f"{pair_stats['color_shift_same_label']:,} → ซ้ำ ลบ | สีต่างและ label ต่าง (คนละระยะสุก) {pair_stats['kept_stage_change']:,} คู่ → เก็บ + flag review")
print(f"[OK] กลุ่มซ้ำ {n_clusters:,} กลุ่ม → flag ลบ {int(df.dup_flag.sum()):,} ภาพ ({ {k: int(v) for k, v in df[df.dup_flag].source.value_counts().items()} }) | "
      f"ซ้ำข้าม source {cross_pairs} คู่ | ภาพที่ติด stage_pair_flag {int(df.stage_pair_flag.sum()):,}")
print(f"[OK] scene groups (≤{D_SCENE}) ที่มี >1 ภาพ: {n_scene:,} กลุ่ม (ไม่ลบ — nb04 จะวางไว้ split เดียวกัน)")
print("     ตัวอย่างชื่อไฟล์เดิมของ Roboflow (ดูว่าชื่อบอกพวง/วันไหม — ใช้วางแผน group ใน nb04):")
for nm in df[df.source == "roboflow"]["orig_name"].sample(min(8, int((df.source == "roboflow").sum())), random_state=SEED):
    print(f"       {nm}")
RUN["thresholds_used"]["dup_hamming"] = {"exact": D_EXACT, "color_rgb": C_MAX, "scene": D_SCENE}
RUN["duplicates"] = {"flagged": int(df.dup_flag.sum()), "clusters": n_clusters, "scene_groups": n_scene, "cross_source_pairs": cross_pairs,
                     "hash_pairs": n_pairs_hash, **{k: int(v) for k, v in pair_stats.items()}, "stage_pair_images": int(df.stage_pair_flag.sum())}

# --- กราฟ distribution ของระยะใกล้สุด ---
fig, ax = plt.subplots(figsize=(8, 3.6))
for src in ("mendeley", "roboflow"):
    vals = df.loc[df.source == src, "nn_dist"].clip(upper=32)
    ax.hist(vals, bins=np.arange(0, 34) - 0.5, alpha=0.7, color=viz.SOURCE_COLORS[src], label=src.capitalize())
ax.axvline(D_EXACT + 0.5, color=viz.TEXT["primary"], ls="--", lw=1.2)
ax.axvline(D_SCENE + 0.5, color=viz.TEXT["secondary"], ls=":", lw=1.2)
ax.text(D_EXACT + 0.7, ax.get_ylim()[1] * 0.9, f"duplicate ≤{D_EXACT}", fontsize=8)
ax.text(D_SCENE + 0.7, ax.get_ylim()[1] * 0.75, f"same scene ≤{D_SCENE}", fontsize=8, color=viz.TEXT["secondary"])
ax.set_title("pHash nearest-neighbour Hamming distance per image", loc="left")
ax.set(xlabel="Hamming distance to nearest image (64-bit pHash)", ylabel="images")
ax.legend()
fig.tight_layout()
out = viz.save_fig(fig, DIRS["figures"], "03_duplicate_distance.png")
plt.show()
plt.close(fig)

# --- ตัวอย่างคู่ซ้ำจริง (ซ้าย = เก็บ, ขวา = ลบ) — 2 คู่ต่อ source ---
pairs = pd.concat([df[df.dup_flag & (df.source == s)].head(2) for s in ("mendeley", "roboflow")])
files, titles = [], []
for r in pairs.itertuples():
    kept = df[(df.exact_cluster == r.exact_cluster) & (~df.dup_flag)].iloc[0]
    files += [kept.file, r.file]
    titles += [f"KEEP {kept.file[:22]} cls={sig[kept.name]}", f"DROP {r.file[:22]} d={r.nn_dist} cls={sig[r.Index]}"]
out2 = image_grid(files, titles, ncols=4, path="03_duplicate_examples.png", suptitle="True duplicates: same structure AND same colour (left kept, right removed)")
# --- ตัวอย่างคู่ "โครงสร้างเหมือน สีต่าง label ต่าง" ที่เก็บไว้ทั้งคู่ (คนละระยะสุก หรือ label noise → ตรวจด้วยตา) ---
files, titles = [], []
for i, j2, dd, cd in kept_stage[:4]:
    files += [df.file.iloc[i], df.file.iloc[j2]]
    titles += [f"KEEP {df.file.iloc[i][:22]} cls={sig.iloc[i]}", f"KEEP {df.file.iloc[j2][:22]} d={dd} colour={cd:.0f} cls={sig.iloc[j2]}"]
out3 = image_grid(files, titles, ncols=4, path="03_same_scene_diff_color_examples.png",
                  suptitle="Same structure, different colour AND different label → both kept (stage change or label noise — review)")
print(f"[OK] {out.name}" + (f", {out2.name}" if out2 else "") + (f", {out3.name}" if out3 else ""))


In [ ]:
# === CELL 5 === ตรวจ label ขั้นละเอียด: bbox เล็กผิดปกติ / annotation ซ้ำ (IoU สูง คลาสเดียวกัน) / ขัดแย้ง (IoU สูง คนละคลาส)
# ทำอะไร : ต่อ bbox: ลบถ้าเล็กกว่า min(p1 ของ area ratio, cap) หรือด้านสั้น < 6 px; ต่อคู่ bbox ในภาพเดียว: IoU>0.9 คลาสเดียวกัน → ลบซ้ำ,
#          IoU>0.7 คนละคลาส → flag review (เก็บไว้ทั้งคู่ แต่บันทึกในรายงาน) ; ภาพที่ไม่เหลือ bbox → ตัดทิ้ง
# ทำไม   : bbox จิ๋ว/ซ้ำทำให้ loss ผิดเพี้ยนและ mAP ต่ำโดยไม่ใช่ความผิดของโมเดล; กล้วยที่ 2 คนติดคนละคลาสคือ label noise ที่ต้องรู้จำนวน
L = NB03["label"]
tiny_p = float(np.percentile(bx["area_ratio"], L["tiny_area_percentile"]))
tiny_thr = min(tiny_p, L["tiny_area_cap"])
print(f"[THRESHOLD] tiny bbox: p{L['tiny_area_percentile']:g} ของ area ratio = {tiny_p:.5f} | cap = {L['tiny_area_cap']} → ใช้ {tiny_thr:.5f} (+ ด้านสั้น < {L['tiny_min_side_px']} px)")
RUN["thresholds_used"]["tiny_area_ratio"] = tiny_thr

def iou_xywh(a, b):
    ax1, ay1, ax2, ay2 = a[0] - a[2] / 2, a[1] - a[3] / 2, a[0] + a[2] / 2, a[1] + a[3] / 2
    bx1, by1, bx2, by2 = b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2
    iw, ih = max(0.0, min(ax2, bx2) - max(ax1, bx1)), max(0.0, min(ay2, by2) - max(ay1, by1))
    inter = iw * ih
    union = a[2] * a[3] + b[2] * b[3] - inter
    return inter / union if union > 0 else 0.0

CLEAN_BOXES, stats, examples = {}, Counter(), {"tiny": [], "dup": [], "conflict": []}
for r in df.itertuples():
    boxes = BOXES[r.file]
    kept = []
    for c, x, y, w, h in boxes:
        if w * h < tiny_thr or min(w * r.width, h * r.height) < L["tiny_min_side_px"]:
            stats["tiny_removed"] += 1
            if len(examples["tiny"]) < 6:
                examples["tiny"].append(r.file)
            continue
        kept.append((c, x, y, w, h))
    # annotation ซ้ำ / ขัดแย้ง (เทียบทุกคู่ — ภาพหนึ่งมีไม่กี่ bbox)
    drop = set()
    conflict = 0
    for i in range(len(kept)):
        for j in range(i + 1, len(kept)):
            if j in drop or i in drop:
                continue
            v = iou_xywh(kept[i][1:], kept[j][1:])
            if v > L["dup_iou"] and kept[i][0] == kept[j][0]:
                drop.add(j if kept[j][3] * kept[j][4] <= kept[i][3] * kept[i][4] else i)   # เก็บกล่องใหญ่กว่า
                stats["dup_box_removed"] += 1
                if len(examples["dup"]) < 6:
                    examples["dup"].append(r.file)
            elif v > L["conflict_iou"] and kept[i][0] != kept[j][0]:
                conflict += 1
                if len(examples["conflict"]) < 6:
                    examples["conflict"].append(r.file)
    kept = [b for k, b in enumerate(kept) if k not in drop]
    if conflict:
        stats["images_with_conflict"] += 1
    CLEAN_BOXES[r.file] = kept
    if not kept:
        stats["images_no_box_left"] += 1

df["n_boxes_clean"] = df["file"].map(lambda f: len(CLEAN_BOXES[f]))
# นับ conflict ต่อภาพแบบเต็ม (examples เก็บแค่ตัวอย่าง)
conf_files = set()
for r in df.itertuples():
    kb = CLEAN_BOXES[r.file]
    if any(iou_xywh(kb[i][1:], kb[j][1:]) > L["conflict_iou"] and kb[i][0] != kb[j][0]
           for i in range(len(kb)) for j in range(i + 1, len(kb))):
        conf_files.add(r.file)
df["label_conflict"] = df["file"].isin(conf_files)
df["label_flag"] = df["n_boxes_clean"] == 0
print(f"[OK] tiny bbox ลบ {stats['tiny_removed']:,} | annotation ซ้ำ ลบ {stats['dup_box_removed']:,} | "
      f"ภาพที่มี bbox ขัดแย้งคนละคลาส (IoU>{L['conflict_iou']}) {len(conf_files):,} (เก็บไว้ + บันทึกใน report) | "
      f"ภาพไม่เหลือ bbox → ตัด {int(df.label_flag.sum()):,}")
RUN["label_checks"] = {**{k: int(v) for k, v in stats.items()}, "images_conflict": len(conf_files), "images_dropped": int(df.label_flag.sum())}

files = examples["tiny"][:3] + examples["dup"][:3] + list(conf_files)[:3]
titles = [f"tiny bbox · {f[:20]}" for f in examples["tiny"][:3]] + [f"dup bbox · {f[:20]}" for f in examples["dup"][:3]] + [f"conflict · {f[:20]}" for f in list(conf_files)[:3]]
out = image_grid(files, titles, ncols=3, path="03_label_issues_examples.png", suptitle="Label issues — original boxes drawn (tiny / duplicated / class-conflict)")
if out:
    print(f"[OK] {out.name}")


In [ ]:
# === CELL 6 === ภาพที่ตัดสินความสุกไม่ได้: กล้วยมืดเกิน / ขาวโพลน (over-exposed) / ซีดเกิน / contrast ต่ำ / กล้วยเล็กเกินในภาพ
# ทำอะไร : ใช้ค่าที่วัดบน "pixel ของวัตถุ" จาก CELL 3: brightness_obj, sat_frac_obj (สัดส่วน pixel ≥250), contrast_obj
#          + max_box_ratio (bbox ใหญ่สุดในภาพ) → threshold ต่อ source แบบ percentile แล้ว cap ด้วยค่ากายภาพ → flag + กราฟ + ตัวอย่าง
# ทำไม   : สีคือ feature หลักของความสุก — กล้วยที่มืด/ขาวโพลนจนสีเพี้ยน หรือเล็กจนเหลือไม่กี่ pixel ตัดสินไม่ได้แม้แต่คน (spec ข้อ 5.4)
E = NB03["exposure"]
df["max_box_ratio"] = df["file"].map(lambda f: max((b[3] * b[4] for b in CLEAN_BOXES[f]), default=0.0))
thr = {}
for src in ("mendeley", "roboflow"):
    g = df[df.source == src]
    thr[src] = {
        "dark": min(float(np.percentile(g.brightness_obj, E["dark_percentile"])), E["dark_cap"]),
        "sat": max(float(np.percentile(g.sat_frac_obj, E["sat_percentile"])), E["sat_floor"]),
        "bright": max(float(np.percentile(g.brightness_obj, E["bright_percentile"])), E["bright_floor"]),
        "contrast": min(float(np.percentile(g.contrast_obj, E["contrast_percentile"])), E["contrast_cap"]),
        "smallbox": min(float(np.percentile(g.max_box_ratio, E["smallest_box_percentile"])), E["smallest_box_cap"]),
    }
    print(f"[THRESHOLD] {src:<9} object brightness < {thr[src]['dark']:.1f} | saturated frac > {thr[src]['sat']:.2f} | object brightness > {thr[src]['bright']:.1f} | "
          f"object contrast < {thr[src]['contrast']:.1f} | largest box < {thr[src]['smallbox']:.4f}")
df["dark_flag"] = df.apply(lambda r: r.brightness_obj < thr[r.source]["dark"], axis=1)
df["bright_flag"] = df.apply(lambda r: r.sat_frac_obj > thr[r.source]["sat"] or r.brightness_obj > thr[r.source]["bright"], axis=1)
df["lowcontrast_flag"] = df.apply(lambda r: r.contrast_obj < thr[r.source]["contrast"], axis=1)
df["smallbox_flag"] = df.apply(lambda r: r.max_box_ratio < thr[r.source]["smallbox"], axis=1)
df["exposure_flag"] = df[["dark_flag", "bright_flag", "lowcontrast_flag", "smallbox_flag"]].any(axis=1)
RUN["thresholds_used"]["exposure"] = thr
RUN["exposure"] = {k: int(df[k].sum()) for k in ("dark_flag", "bright_flag", "lowcontrast_flag", "smallbox_flag", "exposure_flag")}
print(f"[OK] flagged: dark {RUN['exposure']['dark_flag']} | over-exposed {RUN['exposure']['bright_flag']} | low-contrast {RUN['exposure']['lowcontrast_flag']} | "
      f"banana too small {RUN['exposure']['smallbox_flag']} → รวม {RUN['exposure']['exposure_flag']} ภาพ")

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for col, src in enumerate(("mendeley", "roboflow")):
    g = df[df.source == src]
    ax = axes[0, col]
    ax.hist(g.brightness_obj, bins=50, color=viz.SOURCE_COLORS[src], alpha=0.85)
    ax.axvline(thr[src]["dark"], color=viz.TEXT["primary"], ls="--", lw=1.2)
    ax.axvline(thr[src]["bright"], color=viz.TEXT["primary"], ls="--", lw=1.2)
    ax.set_title(f"{src.capitalize()} — mean brightness of object pixels: dark < {thr[src]['dark']:.0f}, washed-out > {thr[src]['bright']:.0f}", loc="left", fontsize=9.5)
    ax.set(xlabel="mean gray level of object pixels (0–255)", ylabel="images")
    ax = axes[1, col]
    ax.hist(g.sat_frac_obj.clip(upper=1), bins=50, color=viz.SOURCE_COLORS[src], alpha=0.85)
    ax.axvline(thr[src]["sat"], color=viz.TEXT["primary"], ls="--", lw=1.2)
    ax.set_yscale("log")
    ax.set_title(f"{src.capitalize()} — saturated fraction of object pixels (≥250): over-exposed > {thr[src]['sat']:.2f}", loc="left", fontsize=9.5)
    ax.set(xlabel="fraction of object pixels ≥ 250", ylabel="images (log)")
fig.tight_layout()
out = viz.save_fig(fig, DIRS["figures"], "03_exposure_distribution.png")
plt.show()
plt.close(fig)

ex = pd.concat([df[df.dark_flag].nsmallest(3, "brightness_obj"), df[df.bright_flag].nlargest(3, "brightness_obj"),
                df[df.lowcontrast_flag].nsmallest(3, "contrast_obj"), df[df.smallbox_flag].nsmallest(3, "max_box_ratio")])
titles = ([f"dark: object mean {r.brightness_obj:.0f}" for r in df[df.dark_flag].nsmallest(3, "brightness_obj").itertuples()]
          + [f"over-exposed: sat {r.sat_frac_obj:.2f} mean {r.brightness_obj:.0f}" for r in df[df.bright_flag].nlargest(3, "brightness_obj").itertuples()]
          + [f"low contrast {r.contrast_obj:.1f}" for r in df[df.lowcontrast_flag].nsmallest(3, "contrast_obj").itertuples()]
          + [f"largest box {r.max_box_ratio:.4f}" for r in df[df.smallbox_flag].nsmallest(3, "max_box_ratio").itertuples()])
out2 = image_grid(ex["file"].tolist(), titles, ncols=6, path="03_exposure_removed_examples.png",
                  boxes_by_file={f: CLEAN_BOXES[f] for f in ex["file"]}, suptitle="Unusable-image check — flagged examples")
print(f"[OK] {out.name}" + (f", {out2.name}" if out2 else ""))


In [ ]:
# === CELL 7 === รวมผล: ก่อน/หลัง cleaning + class balance + เสนอ oversampling → เขียน data/processed + รายงานทั้งหมด
# ทำอะไร : keep = ไม่ติด flag ใด ๆ → copy ภาพ (Mendeley ย่อเหลือด้านยาว 1,280 px) + label ที่ล้าง bbox แล้ว ลง data/processed/
#          สรุปตาราง before/after, กราฟ class balance, คำนวณ oversampling ratio ต่อคลาส, เขียน csv/md/json และอัปเดต project.yaml
# ทำไม   : nb04 ต้องใช้ manifest_clean.csv (มี scene_group กัน leakage) และ nb05 ต้องใช้ oversampling_plan ที่มีเหตุผลรองรับ
FLAGS = ["blur_flag", "dup_flag", "label_flag", "exposure_flag"]
df["keep"] = ~df[FLAGS].any(axis=1)
df["drop_reason"] = df[FLAGS].apply(lambda r: ";".join(k.replace("_flag", "") for k, v in r.items() if v), axis=1)

# --- ตาราง before / after ---
def inst_counts(files):
    c = Counter()
    for f in files:
        for b in CLEAN_BOXES[f]:
            c[b[0]] += 1
    return {CLASSES[k]: int(c.get(k, 0)) for k in CLASSES}
before_img = df.groupby("source").size().to_dict()
after_img = df[df.keep].groupby("source").size().to_dict()
before_inst = {CLASSES[k]: int(v) for k, v in bx["cls"].value_counts().items()}
after_inst = inst_counts(df.loc[df.keep, "file"])
kept = df[df.keep]
print("— BEFORE → AFTER —")
print(f"  images   : {len(df):,} → {len(kept):,}  (removed {len(df) - len(kept):,}: " + ", ".join(f"{k} {v}" for k, v in Counter(sum((r.split(';') for r in df.loc[~df.keep, 'drop_reason']), [])).items()) + ")")
for src in ("mendeley", "roboflow"):
    print(f"    {src:<9}: {before_img.get(src, 0):,} → {after_img.get(src, 0):,}")
print(f"  boxes    : {len(bx):,} → {sum(after_inst.values()):,}")
for name in NAMES:
    print(f"    {name:<10}: {before_inst.get(name, 0):,} → {after_inst[name]:,}")

# --- class balance (image-level dominant class หลัง cleaning) + แผน oversampling ---
kept = kept.copy()
def dominant_of(boxes):
    # คลาสหลักของภาพ = จำนวน bbox มากสุด (เสมอกัน → พื้นที่รวมมากสุด) — กติกาเดียวกับ nb02
    cnt, area = Counter(), defaultdict(float)
    for c, x, y, w, h in boxes:
        cnt[c] += 1
        area[c] += w * h
    return max(cnt, key=lambda c: (cnt[c], area[c]))
kept["dominant_class"] = kept["file"].map(lambda f: dominant_of(CLEAN_BOXES[f]))
img_by_class = {CLASSES[k]: int((kept.dominant_class == k).sum()) for k in CLASSES}
max_inst = max(after_inst.values())
oversample = {name: round(min(NB03["oversample_cap"], max_inst / max(after_inst[name], 1)), 2) for name in NAMES}
imb_ratio = max_inst / max(min(after_inst.values()), 1)
print(f"\n— CLASS BALANCE หลัง cleaning (instance-level) — imbalance ratio max/min = {imb_ratio:.2f}")
print("  คลาส        instances   images(dominant)   oversample×  (ใช้กับ train split เท่านั้น ใน nb04/05)")
for name in NAMES:
    print(f"  {name:<10} {after_inst[name]:>9,} {img_by_class[name]:>15,} {oversample[name]:>12.2f}")
print("  เหตุผล: ratio = max_count / count (cap ×3) — ใช้ 'oversampling' (ทำซ้ำรายการภาพใน train) แทน class-weight เพราะ ultralytics detect loss ไม่มี class_weights;")
print("          บวก targeted augmentation แบบไม่แตะสี (flip/scale/translate) กับคลาสที่ถูก oversample เพื่อไม่ให้ซ้ำภาพเดิมเป๊ะ")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(NC)
bw = 0.38
for ax, (b, a, title) in zip(axes, [(before_inst, after_inst, "Instances per class"),
                                   ({CLASSES[k]: int((df.dominant_class == k).sum()) for k in CLASSES}, img_by_class, "Images per dominant class")]):
    bars1 = ax.bar(x - bw / 2, [b.get(nm, 0) for nm in NAMES], width=bw - 0.02, color="#b9b8b1", label="before cleaning")
    bars2 = ax.bar(x + bw / 2, [a.get(nm, 0) for nm in NAMES], width=bw - 0.02, color=viz.SOURCE_COLORS["mendeley"], label="after cleaning")
    for bars in (bars1, bars2):
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{int(bar.get_height()):,}", ha="center", va="bottom", fontsize=8, color=viz.TEXT["secondary"])
    ax.set_xticks(x, NAMES)
    ax.set_title(title, loc="left", fontsize=11)
    ax.margins(y=0.15)
handles, labels = axes[0].get_legend_handles_labels()
fig.suptitle(f"Class balance before/after cleaning — imbalance ratio {imb_ratio:.2f} (max/min instances)", x=0.01, ha="left", fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.legend(handles, labels, loc="upper right", ncol=2, bbox_to_anchor=(0.99, 0.995))
out = viz.save_fig(fig, DIRS["figures"], "03_class_balance_before_after.png")
plt.show()
plt.close(fig)

# --- เขียน data/processed (ภาพ + label ที่ล้างแล้ว) ---
P = NB03["processed"]
for d in (IM_OUT, LB_OUT):
    shutil.rmtree(d, ignore_errors=True)
    d.mkdir(parents=True, exist_ok=True)
new_size = {}
for r in tqdm(kept.itertuples(), total=len(kept), desc="write processed"):
    src_img, dst_img = IM_IN / r.file, IM_OUT / r.file
    if r.source == "mendeley" and max(r.width, r.height) > P["mendeley_max_side"]:
        with Image.open(src_img) as im:
            s = P["mendeley_max_side"] / max(im.size)
            im = im.convert("RGB").resize((round(im.width * s), round(im.height * s)), Image.LANCZOS)
            im.save(dst_img, quality=P["jpeg_quality"])
            new_size[r.file] = im.size
    else:
        shutil.copy2(src_img, dst_img)
        new_size[r.file] = (r.width, r.height)
    (LB_OUT / r.label).write_text("".join(f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n" for c, x, y, w, h in CLEAN_BOXES[r.file]), encoding="utf-8")
kept["width"] = kept["file"].map(lambda f: new_size[f][0])
kept["height"] = kept["file"].map(lambda f: new_size[f][1])
kept["n_boxes"] = kept["n_boxes_clean"]
for k in CLASSES:
    kept[f"n_{CLASSES[k]}"] = kept["file"].map(lambda f: sum(1 for b in CLEAN_BOXES[f] if b[0] == k))
kept["dominant_name"] = kept["dominant_class"].map(CLASSES)
kept["classes"] = kept["file"].map(lambda f: ";".join(str(c) for c in sorted({b[0] for b in CLEAN_BOXES[f]})))

# verify processed อีกรอบ (เหมือน nb02 CELL 6) ก่อนส่งต่อ
n_img, n_lbl = sum(1 for _ in IM_OUT.iterdir()), sum(1 for _ in LB_OUT.iterdir())
assert n_img == n_lbl == len(kept), f"processed มี {n_img} ภาพ / {n_lbl} label แต่ manifest {len(kept)}"
bad = 0
for lf in LB_OUT.glob("*.txt"):
    rows = parse_label(lf)
    bad += (not rows) + sum(1 for c, x, y, w, h in rows if c not in CLASSES or not all(0 <= v <= 1 for v in (x, y, w, h)) or w <= 0 or h <= 0)
assert bad == 0, f"processed labels มีปัญหา {bad} รายการ"
size_gb = sum(p.stat().st_size for p in IM_OUT.iterdir()) / 1e9
print(f"[OK] data/processed: {n_img:,} ภาพ ({size_gb:.2f} GB) | Mendeley ย่อเหลือด้านยาว ≤ {P['mendeley_max_side']} px | labels verified")

# --- รายงาน: manifest_clean.csv / data_cleaning_report.csv / before_after_summary.md / json / project.yaml ---
mcols = ["file", "label", "source", "orig_split", "orig_name", "group_id", "scene_group", "width", "height", "n_boxes", "classes",
         "dominant_class", "dominant_name", *[f"n_{n}" for n in NAMES], "blur_ratio", "brightness_obj", "contrast_obj", "sat_frac_obj", "phash", "label_conflict", "stage_pair_flag"]
manifest_clean = DIRS["reports"] / "manifest_clean.csv"
kept[mcols].to_csv(manifest_clean, index=False, encoding="utf-8-sig")
rcols = ["file", "source", "orig_name", "width", "height", "n_boxes", "n_boxes_clean", "vol", "vol_box", "blur_ratio", "brightness", "obj_frac",
         "brightness_obj", "contrast_obj", "sat_frac_obj", "max_box_ratio",
         "nn_dist", "nn_file", "exact_cluster", "scene_group", "stage_pair_flag", *FLAGS, "dark_flag", "bright_flag", "lowcontrast_flag", "smallbox_flag",
         "label_conflict", "keep", "drop_reason"]
report_csv = DIRS["reports"] / "data_cleaning_report.csv"
df[rcols].to_csv(report_csv, index=False, encoding="utf-8-sig")

removed_by = Counter(sum((r.split(";") for r in df.loc[~df.keep, "drop_reason"]), []))
md = [f"# Data cleaning — before / after summary", f"generated by 03_data_cleaning_eda.ipynb on {datetime.now():%Y-%m-%d %H:%M}", "",
      "## Thresholds used (percentile of each source, capped)", "",
      f"- blur: in-box blur ratio (VoL / VoL after σ={NB03['blur']['sigma']} blur @640px) < {json.dumps({k: round(v, 2) for k, v in blur_thr.items()})} (p{NB03['blur']['percentile']:g}, cap {NB03['blur']['ratio_cap']})",
      f"- duplicates: pHash Hamming ≤ {D_EXACT} AND (mean object colour distance ≤ {C_MAX:g} OR same label signature) → removed, keep sharpest "
      f"(same-colour pairs {pair_stats['same_color']}, colour-shifted copies with unchanged label {pair_stats['color_shift_same_label']}); "
      f"same structure + different colour + different label → both kept & flagged ({pair_stats['kept_stage_change']} pairs) · same scene ≤ {D_SCENE} grouped for split",
      f"- tiny bbox: area ratio < {tiny_thr:.5f} or short side < {L['tiny_min_side_px']} px · duplicate bbox IoU > {L['dup_iou']} · conflict IoU > {L['conflict_iou']}",
      f"- exposure (measured on object pixels = box minus background colour): " + "; ".join(f"{s}: brightness<{t['dark']:.0f} or >{t['bright']:.0f}, saturated frac>{t['sat']:.2f}, contrast<{t['contrast']:.1f}, largest box<{t['smallbox']:.4f}" for s, t in thr.items()),
      "", "## Images", "", "| | before | after | removed |", "|---|---|---|---|",
      f"| total | {len(df):,} | {len(kept):,} | {len(df) - len(kept):,} |",
      *[f"| {s} | {before_img.get(s, 0):,} | {after_img.get(s, 0):,} | {before_img.get(s, 0) - after_img.get(s, 0):,} |" for s in ("mendeley", "roboflow")],
      "", "Removed by reason (an image can have several): " + ", ".join(f"{k} {v}" for k, v in removed_by.items()),
      f"\nbbox removed inside kept images: tiny {stats['tiny_removed']}, duplicate annotation {stats['dup_box_removed']}; images with class-conflict boxes kept for review: {len(conf_files)}",
      "", "## Instances per class", "", "| class | before | after | images (dominant) | oversample × |", "|---|---|---|---|---|",
      *[f"| {n} | {before_inst.get(n, 0):,} | {after_inst[n]:,} | {img_by_class[n]:,} | {oversample[n]:.2f} |" for n in NAMES],
      "", f"Imbalance ratio (max/min instances) after cleaning: **{imb_ratio:.2f}**. Recommended: oversample minority classes in the *train split only* "
      f"(ratios above, cap ×{NB03['oversample_cap']:g}) plus colour-preserving augmentation (flip / scale / translate); no class weights (not supported by ultralytics detect loss).",
      "", f"Mendeley images resized to max side {P['mendeley_max_side']} px (labels are normalised → unchanged). EXIF orientation already applied in nb02.",
      "", "## Figures", "", *[f"- figures/{p.name}" for p in sorted(DIRS["figures"].glob("03_*.png"))]]
summary_md = DIRS["reports"] / "before_after_summary.md"
summary_md.write_text("\n".join(md) + "\n", encoding="utf-8")

RUN.update({"before": {"images": int(len(df)), "by_source": {k: int(v) for k, v in before_img.items()}, "instances": before_inst},
            "after": {"images": int(len(kept)), "by_source": {k: int(v) for k, v in after_img.items()}, "instances": after_inst,
                      "images_by_dominant_class": img_by_class},
            "removed_by_reason": dict(removed_by), "imbalance_ratio": round(imb_ratio, 3), "oversampling_plan": oversample,
            "processed_size_gb": round(size_gb, 3), "finished_at": datetime.now().isoformat(timespec="seconds")})
with open(DIRS["reports"] / "03_cleaning_summary.json", "w", encoding="utf-8") as f:
    json.dump(RUN, f, ensure_ascii=False, indent=2, default=str)
CONFIG["datasets"]["actual_after_cleaning"] = {"total": int(len(kept)), **{k: int(v) for k, v in after_img.items()},
                                               "instances": after_inst, "imbalance_ratio": round(imb_ratio, 3)}
CONFIG["oversampling_plan"] = oversample
CONFIG["cleaning_thresholds"] = RUN["thresholds_used"]
with open(CFG_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(CONFIG, f, allow_unicode=True, sort_keys=False)

N = len(kept)
ratio = CONFIG["datasets"]["split_ratio"]
pt, pv = round(N * ratio["train"]), round(N * ratio["valid"])
print(f"\n[OK] {manifest_clean.relative_to(PROJECT_ROOT)} ({len(kept):,} แถว) | {report_csv.relative_to(PROJECT_ROOT)} ({len(df):,} แถว)")
print(f"[OK] {summary_md.relative_to(PROJECT_ROOT)} | reports/03_cleaning_summary.json | project.yaml → actual_after_cleaning + oversampling_plan")
print("\n=== SUMMARY · Notebook 03 ===")
print(f"  {len(df):,} → {N:,} ภาพ (ตัด {len(df) - N:,}) | boxes {len(bx):,} → {sum(after_inst.values()):,} | imbalance {imb_ratio:.2f}")
print(f"  split 80/10/10 ที่คาดสำหรับ nb04: train {pt:,} / valid {pv:,} / test {N - pt - pv:,}")
print(f"  ถัดไป: 04_split_dataset.ipynb")


## ผลลัพธ์ที่ควรเห็น
- CELL 3–6 พิมพ์ `[THRESHOLD]` ทุกเกณฑ์พร้อมบอกว่า percentile หรือ cap "ชนะ" — ตัวเลขนี้ต้องใส่ในเล่มรายงาน
- กราฟ 8–9 รูปใน `figures/03_*.png`: EDA overview, samples grid, blur distribution + ตัวอย่างที่ตัด, duplicate distance + คู่ซ้ำ, label issues, exposure + ตัวอย่าง, class balance ก่อน/หลัง
- CELL 7: ตาราง BEFORE → AFTER, oversampling ratio ต่อคลาส, `data/processed/` เล็กกว่า interim มาก (Mendeley ย่อแล้ว)
- ไฟล์รายงาน: `reports/data_cleaning_report.csv` (ทุกภาพ + ทุก flag + เหตุผลที่ตัด), `reports/before_after_summary.md`, `reports/manifest_clean.csv`

## ปัญหาที่อาจเจอ + วิธีแก้
| อาการ | สาเหตุ | วิธีแก้ |
|---|---|---|
| ตัดภาพเยอะเกินคาด (>10%) | percentile ชนะ cap ทุกเกณฑ์ | ดู `[THRESHOLD]` ว่าเกณฑ์ไหนตัดเยอะ → ลด percentile ใน `NB03` (เช่น blur 3 → 1) แล้ว Run All ใหม่ |
| blur ตัด Mendeley เยอะ | — | ไม่ควรเกิดแล้ว: ใช้ blur ratio ในกล่อง (VoL ทั้งภาพจะตัด unripe ผิวเรียบทิ้งอย่างลำเอียง — ตรวจพบและแก้แล้ว) |
| ตัดน้อยผิดปกติ / 0 ภาพ | cap ชนะทุกเกณฑ์ = ชุดข้อมูลคุณภาพดี | ปกติ — รายงานว่า cap ชนะ (เป็นข้อค้นพบ ไม่ใช่ปัญหา) |
| CELL 4 ช้า (> 5 นาที) | ภาพ > 20k | ลด `hash_size` ไม่ได้ช่วย — ใช้เท่านี้ได้ (5k ภาพ ≈ 10–20 วินาที) |
| CELL 3 คำนวณใหม่ (~2 นาที) แม้เคยรันแล้ว | เพิ่ม metric สีวัตถุ (v4) → cache เก่าไม่ครบคอลัมน์ | ปกติ รันครั้งเดียว |
| CELL 4 `สีถูกเลื่อนแต่ label เดิม (สำเนา)` มีจำนวนมาก | ชุดข้อมูลต้นทางมีสำเนาภาพที่ถูกปรับสี/white balance ติดมา | ถูกต้องที่ลบ — สำเนาที่สีถูกเลื่อนโดย label ไม่เปลี่ยน เป็นพิษกับงานที่ใช้สีตัดสิน; ภาพต้นฉบับยังอยู่ |
| CELL 4 `stage_pair_flag` > 0 | ภาพเดิมแต่ label ต่างกัน | ดูรูป `03_same_scene_diff_color_examples.png` — ถ้าเป็น label noise ให้ส่งรายชื่อจาก `data_cleaning_report.csv` มาแก้ใน nb04 |
| `manifest ... แต่ interim มี ... ไฟล์` | รัน nb02 ไม่จบ | Run All nb02 ใหม่ |
